# Lab 1 — 몬티 홀 문제 (Monty Hall Problem)

**확률통계 · Week 1 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 게임의 **규칙을 코드로 그대로 옮긴다.**
2. 10,000번 반복해서 두 전략(유지 / 교체)의 **승률을 비교**한다.
3. 시행 횟수를 늘릴수록 추정값이 **안정되는 것**을 확인한다.

⏱ **예상 소요 시간: 35분**

---

### 문제

- 문이 **3개** 있다. 그중 **하나 뒤에만 상품**이 있다.
- 당신이 문 하나를 고른다.
- 진행자는 **상품이 없고 당신이 고르지도 않은 문**을 열어서 보여준다.
- 이제 진행자가 묻는다. **"바꾸시겠습니까?"**

> **바꾸는 것이 유리한가, 그대로 가는 것이 유리한가, 아니면 상관없는가?**

먼저 **직접 답을 정해놓고** 시작하자. 아래 셀을 편집해서 자기 예상을 적어두면 된다.

**나의 예상:** (여기에 작성 — 예: "상관없다. 둘 다 1/2")

## Part 0. 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)   # 이 과목 공통 시드
print("준비 완료")

## Part 0.5. List Comprehension 이해하기

곧 나올 **실습 1** 은 이런 모양의 문법을 쓴다.

```python
결과 = [담을_값 for 하나 in 목록 if 조건]
```

처음 보면 낯설지만 **새로운 기능이 아니다.** `for` 문으로 리스트를 만드는 것을
**한 줄로 줄여 쓴 것**뿐이다. 아래 셀들을 그냥 **실행해서** 눈으로 확인하자.
고칠 곳은 없다.

> 여기서는 몬티 홀과 무관한 **시험 점수**로 연습한다. 실습 1 은 직접 조합해 볼 것.

In [ ]:
scores = [88, 45, 92, 61, 73, 100, 39, 58]   # 8명의 점수
print("점수:", scores)

# ── 1) for 로 하나씩 확인하기
print("\n1) for 로 하나씩")
for s in scores:
    print("   ", s, "→ 합격" if s >= 60 else "→ 불합격")

# ── 2) 조건에 맞는 것만 골라 담기  (for + if + append)
passed = []
for s in scores:
    if s >= 60:
        passed.append(s)
print("\n2) for + if + append")
print("   60점 이상:", passed)

# ── 3) 같은 일을 한 줄로  (list comprehension)
passed2 = [s for s in scores if s >= 60]
print("\n3) 한 줄로 쓰면")
print("   60점 이상:", passed2)
print("   2)와 결과가 같은가?", passed == passed2)

읽는 순서가 중요하다. **왼쪽부터가 아니라 가운데부터** 읽는다.

```python
[  s        for s in scores    if s >= 60  ]
#  ③ 담을 값   ① 하나씩 꺼내서    ② 이 조건을 통과하면
```

① `scores` 에서 `s` 를 하나씩 꺼낸다 → ② 조건을 통과하는 것만 → ③ 그 값을 리스트에 담는다.

**조건은 여러 개**를 `and` 로 묶을 수 있고, **담을 값**도 꺼낸 것 그대로일 필요가 없다.

In [ ]:
# 조건을 두 개 — and 로 묶는다
mid = [s for s in scores if s >= 60 and s < 90]
print("60 이상이면서 90 미만:", mid)

# 담을 값을 바꾸기 — 5점씩 올려서 담는다 (최대 100)
bonus = [min(s + 5, 100) for s in scores]
print("5점 가산            :", bonus)

# 조건 없이 값만 바꾸기
half = [s / 2 for s in scores]
print("절반으로            :", half)

### 왜 이 과목에서 자주 쓰나

**True / False 를 담은 리스트는 평균이 곧 비율**이기 때문이다.
"몇 번 중 몇 번"을 세는 일이 이 과목의 거의 전부라, 이 한 줄이 계속 나온다.

In [ ]:
is_pass = [s >= 60 for s in scores]    # True / False 를 담는다
print("합격 여부:", is_pass)

print(f"합격률 = {np.mean(is_pass):.4f}   ({sum(is_pass)}명 / {len(scores)}명)")

> 💡 `np.mean` 은 True 를 1, False 를 0 으로 세어 평균을 낸다. 그래서 **평균 = 비율**이다.
> **실습 3** 에서 승률을 구할 때 이 성질을 그대로 쓴다.

이제 준비가 끝났다. **실습 1 은 조건이 두 개**다.
위에서 본 `and` 로 묶는 법과, 같지 않다를 뜻하는 `!=` 를 **직접 조합**해 보자.

## Part 1. 게임 한 판을 코드로 옮기기

규칙을 **한 줄씩 그대로** 코드로 만든다. 함수 하나가 게임 **한 판**이다.

- `prize` : 상품이 있는 문 번호
- `choice` : 내가 처음 고른 문 번호
- `can_open` : 진행자가 열 수 있는 문 = **상품이 아니고** + **내가 고른 것도 아닌** 문
- `opened` : 진행자가 실제로 연 문들 (문이 $n$개면 $n-2$개를 연다)
- `remaining` : 열리지 않고 남은 문 = 바꿀 수 있는 문 (항상 1개)

### ✏️ 실습 1 — `can_open` 을 채우세요
### ✏️ 실습 2 — 교체(`switch=True`)일 때 문을 바꾸세요

In [ ]:
def play_once(switch, rng, n_doors=3):
    doors = range(n_doors)
    prize = rng.integers(0, n_doors)     # 상품 위치
    choice = rng.integers(0, n_doors)    # 내 첫 선택

    # TODO 1: 진행자가 열 수 있는 문의 리스트를 만드세요
    #         조건 - 상품이 있는 문이 아니고(d != prize), 내가 고른 문도 아니다(d != choice)
    can_open = [d for d in doors]        # <- 조건을 추가하세요

    # 그중 n_doors - 2 개를 연다
    opened = set(rng.choice(can_open, size=n_doors - 2, replace=False).tolist())

    # 열리지 않고 남은 문 (내 선택 제외) - 항상 1개다
    remaining = [d for d in doors if d != choice and d not in opened]

    if switch:
        pass    # TODO 2: 교체 전략이면 choice를 remaining[0] 으로 바꾸세요

    return choice == prize      # 이겼으면 True


# 한 판 해보기
print(play_once(switch=True, rng=rng))

> 💡 방금 쓴 `[d for d in doors if 조건]` 이 **Part 0.5** 에서 본 그 문법이다.
> 막혔다면 거기 3)번 줄을 다시 볼 것.

### ✏️ 실습 3 — 10,000번 반복해서 승률 구하기

`play_once` 는 True/False를 돌려준다. True의 **비율**이 곧 승률이다.

In [ ]:
def win_rate(switch, n_trials=10000, n_doors=3, seed=20260302):
    rng = np.random.default_rng(seed)
    wins = [play_once(switch, rng, n_doors) for _ in range(n_trials)]
    # TODO 3: wins 안의 True 비율(= 승률)을 돌려주세요.  힌트: np.mean
    return 0.0


rate_stay = win_rate(switch=False)
rate_switch = win_rate(switch=True)

print(f"유지(stay)  승률: {rate_stay:.4f}")
print(f"교체(switch) 승률: {rate_switch:.4f}")

처음에 적어둔 예상과 맞았는가?

### ✏️ 실습 4 — 그림으로 비교하기

두 전략의 승률을 막대그래프로 그려보자.

> 📌 그래프의 축 이름과 제목은 **영어로** 쓴다. Colab에는 한글 폰트가 없어 글자가 깨진다.

In [ ]:
plt.figure(figsize=(5, 3.5))
# TODO 4: 두 전략의 승률을 막대그래프로 그리세요
#         힌트: plt.bar(["Stay", "Switch"], [rate_stay, rate_switch])

plt.axhline(1 / 3, linestyle="--", color="black", linewidth=1)
plt.ylim(0, 1)
plt.ylabel("Winning probability")
plt.title("Monty Hall: 3 doors, 10000 trials")
plt.show()

## Part 2. 문을 100개로 늘리면?

문이 3개일 때는 차이가 잘 안 느껴질 수 있다. **100개**로 늘려보자.

- 문 100개 중 하나를 고른다 (맞을 확률 1/100)
- 진행자가 **98개**를 열어 보여준다
- 남은 문은 내 문 하나, 그리고 다른 문 하나

In [ ]:
rate_stay_100 = win_rate(switch=False, n_trials=5000, n_doors=100)
rate_switch_100 = win_rate(switch=True, n_trials=5000, n_doors=100)

print(f"문 100개 - 유지  승률: {rate_stay_100:.4f}")
print(f"문 100개 - 교체 승률: {rate_switch_100:.4f}")

🤔 문이 100개일 때는 직관적으로도 납득이 되지 않는가?

> 진행자가 98개를 열어젖히는 동안 **내 문은 한 번도 후보에서 검토되지 않았다.**
> 진행자는 답을 알고 있고, 남은 한 문에 **정보를 몰아준 것**이다.

이것이 왜 확률을 바꾸는지는 **2주차 조건부 확률**에서 정확히 계산한다.

## Part 3. 몇 번 돌려야 믿을 수 있나

오늘 강의 도입부의 질문이 여기서도 똑같이 나온다.
**시행 횟수를 늘려가며 추정 승률이 어떻게 안정되는지** 보자.

### ✏️ 실습 5

In [ ]:
rng = np.random.default_rng(20260302)

N = 5000
wins = np.array([play_once(True, rng) for _ in range(N)])
n = np.arange(1, N + 1)

running = np.zeros(N)   # TODO 5: 처음 n판까지의 누적 승률로 바꾸세요 (np.cumsum 사용)

plt.figure(figsize=(7, 3.5))
plt.plot(n, running, label="switch")
plt.axhline(2 / 3, color="red", linestyle="--", label="2/3")
plt.ylim(0, 1)
plt.xlabel("Number of games")
plt.ylabel("Estimated win rate")
plt.title("How many games are enough?")
plt.legend()
plt.show()

❓ **100판만 돌리고 "교체가 유리하다"고 결론 내려도 될까?**

이 질문의 정확한 답 — *"오차가 얼마인지, 몇 번이면 충분한지"* — 는 **10주차**에서 한다.

---

## 부록 A. 순열과 조합 (과제 1 준비)

강의에서는 다루지 않았다. **코드로 하면 된다.**

In [ ]:
import math

# 순열 P(n, r) : n개에서 r개를 뽑아 순서대로 늘어놓는 경우의 수
print("P(5,3) =", math.perm(5, 3))     # 5 x 4 x 3 = 60

# 조합 C(n, r) : n개에서 r개를 순서 없이 고르는 경우의 수
print("C(5,3) =", math.comb(5, 3))     # 10

# 팩토리얼
print("5! =", math.factorial(5))

# 예) 생일 문제에서 쓰는 계산 - n명의 생일이 모두 다를 경우의 수
n = 23
distinct = math.perm(365, n)           # 365 x 364 x ... x (365-n+1)
total = 365 ** n
print(f"{n}명의 생일이 모두 다를 확률: {distinct / total:.4f}")

> 위 마지막 줄을 보면 과제 1의 절반은 이미 끝난 셈이다.
> **여사건**(모두 다르다)을 계산한 뒤 1에서 빼면 "적어도 두 명이 같다"가 나온다.

## 부록 B. 그리스 문자 읽는 법

수식에 나오는 문자를 읽지 못하면 수업을 따라가기 어렵다. 자주 나오는 것만.

| 문자 | 읽기 | 이 과목에서의 쓰임 |
|---|---|---|
| $\Omega,\ \omega$ | 오메가 | 표본공간 |
| $\xi$ | 크사이 | 하나의 결과(outcome) |
| $\mu$ | 뮤 | 평균 |
| $\sigma,\ \Sigma$ | 시그마 | 표준편차 / 합 |
| $\lambda$ | 람다 | Poisson·Exponential의 모수 |
| $\theta$ | 쎄타 | 추정하려는 모수 (12주차) |
| $\alpha,\ \beta$ | 알파, 베타 | 유의수준 / Beta 분포 |

---

## 마무리 — 자가 점검

- [ ] 게임 규칙을 코드로 옮길 수 있었다
- [ ] 유지 1/3, 교체 2/3 라는 결과를 직접 확인했다
- [ ] 문이 100개일 때 왜 더 분명해지는지 말로 설명할 수 있다
- [ ] 시행 횟수가 적을 때 추정값이 흔들리는 것을 보았다

**오늘 가장 놀라웠던 점을 한 문장으로 써보자.**

> (여기에 작성)

---

### 📌 과제 1 — 생일 문제

`hw/W01_hw.md` 참고. 부록 A의 코드가 출발점이다. **다음 주 수업 전까지 제출.**